# Task 3: Molecular Fingerprints — MACCS Keys
**Method:** MACCS Keys (166-bit structural key dictionary)  
**Tool:** RDKit — `rdkit.Chem.MACCSkeys.GenMACCSKeys()`

### Why MACCS Keys?
| Criterion | Rationale |
|----------|----------|
| **Interpretable** | Each of 166 bits = a defined chemical feature (e.g., aromatic ring, NH group) |
| **Standardized** | SMARTS-based dictionary — same bit always means the same thing |
| **Open-source** | Fully supported in RDKit (no license needed) |
| **Literature** | Widely validated in QSAR, virtual screening, scaffold analysis |
| **Compact** | 166 bits avoids high-dimensionality issues on small datasets |
| **Similarity** | Tanimoto on MACCS is well-validated for scaffold-level similarity |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import MACCSkeys
    RDKIT = True
    print('✅ RDKit loaded')
except ImportError:
    RDKIT = False
    print('⚠️  RDKit not available — loading pre-computed fingerprints')

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('../data/bioactivity_dataset.csv')
df['standard_value'] = df['standard_value'].replace(0, 0.001)
df['pIC50'] = (9 - np.log10(df['standard_value'])).round(4)
print(f'Loaded {len(df)} compounds')

## 3.1 Generate MACCS Keys Fingerprints

In [ ]:
if RDKIT:
    def smiles_to_maccs(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return [0] * 166
        fp = MACCSkeys.GenMACCSKeys(mol)
        return list(fp)[1:]  # 167 bits; bit 0 always 0

    print('Computing MACCS fingerprints...')
    fps = [smiles_to_maccs(s) for s in df['smiles']]
    fp_cols = [f'MACCS_{i+1}' for i in range(166)]
    fp_df = pd.DataFrame(fps, columns=fp_cols)
    fp_df.insert(0, 'chembl_id', df['chembl_id'])
    fp_df.insert(1, 'pIC50', df['pIC50'])
    fp_df.insert(2, 'bioactivity_class', df['bioactivity_class'])
    fp_df.to_csv('../data/maccs_fingerprints.csv', index=False)
    print(f'✅ MACCS fingerprints saved: {fp_df.shape}')
else:
    fp_df = pd.read_csv('../data/maccs_fingerprints.csv')
    print(f'✅ Pre-computed fingerprints loaded: {fp_df.shape}')

fp_df.iloc[:5, :8]

## 3.2 Fingerprint Statistics

In [ ]:
fp_mat = fp_df.iloc[:, 3:].values
print(f'Fingerprint matrix: {fp_mat.shape}')
print(f'Avg bits set per compound: {fp_mat.sum(axis=1).mean():.1f} / 166')
print(f'Average bit density: {fp_mat.mean()*100:.1f}%')

print('\nBits set by class:')
for cls in ['active','inactive','intermediate']:
    mask = fp_df['bioactivity_class'] == cls
    avg = fp_mat[mask].sum(axis=1).mean()
    print(f'  {cls:15s}: {avg:.1f} bits/molecule')

## 3.3 Fingerprint Visualizations

In [ ]:
colors = {'active': '#2ecc71', 'inactive': '#e74c3c', 'intermediate': '#f39c12'}
bit_freq = fp_mat.mean(axis=0) * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('MACCS Keys Fingerprint Analysis — 6,843 Compounds', fontsize=15, fontweight='bold')

# Overall bit frequency
axes[0,0].bar(range(1, 167), bit_freq, color='#8e44ad', alpha=0.75, width=1.0)
axes[0,0].set_xlabel('MACCS Key Index', fontsize=11)
axes[0,0].set_ylabel('Frequency (%)', fontsize=11)
axes[0,0].set_title('Bit Frequency — All Compounds', fontsize=12, fontweight='bold')
axes[0,0].grid(axis='y', alpha=0.3)

# Bit freq by class
for cls in ['active','inactive','intermediate']:
    mask = fp_df['bioactivity_class'] == cls
    freq = fp_mat[mask].mean(axis=0) * 100
    axes[0,1].plot(range(1, 167), freq, alpha=0.7, lw=1.3, color=colors[cls], label=cls)
axes[0,1].set_xlabel('MACCS Key Index', fontsize=11)
axes[0,1].set_ylabel('Frequency (%)', fontsize=11)
axes[0,1].set_title('Bit Frequency by Class', fontsize=12, fontweight='bold')
axes[0,1].legend(fontsize=10)
axes[0,1].grid(alpha=0.3)

# Tanimoto similarity
def tanimoto(a, b):
    ab = float(np.dot(a, b))
    return ab / (a.sum() + b.sum() - ab + 1e-9)

np.random.seed(0)
sample_idx = np.random.choice(len(fp_mat), 150, replace=False)
fp_s = fp_mat[sample_idx].astype(float)
cls_s = fp_df['bioactivity_class'].values[sample_idx]
same, diff = [], []
for i in range(100):
    for j in range(i+1, 100):
        t = tanimoto(fp_s[i], fp_s[j])
        (same if cls_s[i]==cls_s[j] else diff).append(t)
axes[1,0].hist(same, bins=30, alpha=0.6, color='#3498db', label='Same class', edgecolor='none')
axes[1,0].hist(diff, bins=30, alpha=0.6, color='#e74c3c', label='Diff. class', edgecolor='none')
axes[1,0].set_xlabel('Tanimoto Similarity', fontsize=11)
axes[1,0].set_ylabel('Frequency', fontsize=11)
axes[1,0].set_title('Pairwise Tanimoto Similarity', fontsize=12, fontweight='bold')
axes[1,0].legend(fontsize=10)
axes[1,0].grid(alpha=0.3)

# Bits set per compound
bpc = fp_mat.sum(axis=1)
for cls in ['active','inactive','intermediate']:
    mask = fp_df['bioactivity_class'] == cls
    axes[1,1].hist(bpc[mask], bins=30, alpha=0.55, color=colors[cls],
                   label=f'{cls} (μ={bpc[mask].mean():.1f})')
axes[1,1].set_xlabel('Bits Set per Compound', fontsize=11)
axes[1,1].set_ylabel('Count', fontsize=11)
axes[1,1].set_title('Bits Set per Compound by Class', fontsize=12, fontweight='bold')
axes[1,1].legend(fontsize=10)
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/task3_maccs_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# MACCS heatmap (top 40 bits × first 100 compounds)
top_40 = np.argsort(bit_freq)[-40:][::-1]
hmap   = fp_mat[:100, top_40]
labels = [f'M{b+1}' for b in top_40]

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(hmap, ax=ax, cmap='RdYlGn', xticklabels=labels, yticklabels=False,
            linewidths=0.15, linecolor='white', cbar_kws={'label': 'Bit (1=present)'})
ax.set_title('MACCS Keys Heatmap — Top 40 Bits × First 100 Compounds', fontsize=13, fontweight='bold')
ax.set_xlabel('MACCS Key', fontsize=11)
ax.set_ylabel('Compounds', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/task3_maccs_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Task 3 Complete!')